## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; RAW_DATA_DIR=DATA_DIR/"01_raw"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"
MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

# 08 · Risk scoring

A model probability is a ranking signal. A useful risk score also needs a clear scale, calibration checks, and documented limits. This notebook loads the selected artifact, reproduces the chronological validation split, scores it, and creates illustrative risk bands. The bands are examples for analysis, not universal operating thresholds.

## 1. Load engineered data and the selected model

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

feature_path=INTERMEDIATE_DIR/"train_features.pkl"; artifact_path=MODEL_DIR/"best_experiment_model.joblib"
if not feature_path.exists() or not artifact_path.exists(): raise FileNotFoundError("Run notebooks 04 and 06 first.")
train_fe=pd.read_pickle(feature_path); artifact=joblib.load(artifact_path); feature_cols=list(artifact["feature_cols"])
missing=sorted(set(feature_cols)-set(train_fe.columns))
if missing: raise ValueError(f"Feature data lacks model inputs: {missing[:10]}")
print(f"Model: {artifact.get('experiment_name',type(artifact['model']).__name__)} | features: {len(feature_cols)} | rows: {len(train_fe):,}")

## 2. Rebuild the chronological validation window

The latest 20% of training events form validation, matching notebooks 05–07. The same window influenced model selection and threshold choices, so these numbers are exploratory rather than an unbiased final estimate.

In [ ]:
ordered=train_fe.sort_values("TransactionDT",kind="stable").reset_index(drop=True)
val=ordered.iloc[int(len(ordered)*0.8):].copy()
X=val[feature_cols].replace([np.inf,-np.inf],np.nan); y=val["isFraud"].astype(int)
model=artifact["model"]; imputer=artifact.get("imputer"); scaler=artifact.get("scaler")
if imputer is not None: X=imputer.transform(X)
if scaler is not None: X=scaler.transform(X)
probability=model.predict_proba(X)[:,1]
print(f"Rows: {len(y):,} | fraud rate: {y.mean():.3%}")
print(f"ROC-AUC: {roc_auc_score(y,probability):.4f} | PR-AUC: {average_precision_score(y,probability):.4f}")

## 3. Convert probability to a readable score

The 0–100 score is the model output multiplied by 100. It does not make an uncalibrated model calibrated. Check notebook 07’s reliability curve before describing it as a literal probability.

In [ ]:
scored=val[[c for c in ["TransactionID","TransactionDT","TransactionAmt","isFraud"] if c in val]].copy()
scored["fraud_probability"]=probability
scored["risk_score"]=(100*probability).round(2)
scored.head()

## 4. Use illustrative bands and inspect observed outcomes

In [ ]:
BANDS=[0.05,0.20]  # illustrative only; choose using validated costs and review capacity
scored["risk_band"]=pd.cut(scored["fraud_probability"],bins=[-np.inf,*BANDS,np.inf],labels=["lower","elevated","high"],right=False)
summary=scored.groupby("risk_band",observed=False).agg(transactions=("risk_score","size"),mean_score=("risk_score","mean"),observed_fraud_rate=("isFraud","mean"),fraud_cases=("isFraud","sum"))
summary

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
axes[0].hist(scored["risk_score"],bins=30,color="slateblue"); axes[0].set(title="Validation score distribution",xlabel="Score (0–100)",ylabel="Transactions")
summary["observed_fraud_rate"].plot.bar(ax=axes[1],color="firebrick"); axes[1].set(title="Observed fraud rate by band",xlabel="Illustrative band",ylabel="Fraud rate")
plt.tight_layout(); plt.show()

## 5. Interpretation limits

- Historical benchmark output is not a guarantee about a future transaction.
- Labels may arrive late or be incomplete, changing measured fraud rates.
- Cost, review capacity, customer impact, and calibration matter when choosing actions.
- These engineered velocity features are offline. Live inference needs stateful features using only past events.